# 02 — Train YOLOv8s on FLIR Thermal Dataset

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# Cell 1 — Install ultralytics
%pip install ultralytics -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Cell 2 — Config (edit only this cell)
import torch
from ultralytics import YOLO

MODEL   = "yolov8s.pt"                        # YOLOv8 small — downloads automatically
DATA    = r"D:\SEM 6\AIP\Project\data\processed\dataset.yaml"
EPOCHS  = 45
BATCH   = 8
IMGSZ   = 512
DEVICE  = 0 if torch.cuda.is_available() else "cpu"   # GPU if available, else CPU
PROJECT = "../runs/train"
NAME    = "night_vision_v1"

print(f"Using device: {'GPU (cuda:' + str(DEVICE) + ')' if isinstance(DEVICE, int) else 'CPU'}")
if isinstance(DEVICE, int):
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

Using device: GPU (cuda:0)
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
# Cell 3 — Load model and train
# YOLOv8 downloads yolov8s.pt automatically on first run
model = YOLO(MODEL)

results = model.train(
    data    = DATA,
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = DEVICE,
    project = PROJECT,
    name    = NAME,
    workers=1,
    cache=False,
    persistent_workers=False

)

print("\nTraining complete!")
print(f"Run saved to: {results.save_dir}")

Ultralytics 8.4.33  Python-3.11.0 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\SEM 6\AIP\Project\data\processed\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=45, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=night_vision_v15, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

In [ ]:
# Cell 4 — Copy best.pt to models/best.pt
import os
import shutil

best_src = os.path.join(str(results.save_dir), "weights", "best.pt")
best_dst = "../models/best.pt"

os.makedirs("../models", exist_ok=True)
shutil.copy2(best_src, best_dst)

print(f"Copied: {best_src}")
print(f"    ->  {os.path.abspath(best_dst)}")

In [ ]:
# Cell 5 — Display results.png (YOLOv8 auto-saves loss curves + metrics here)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_png = os.path.join(str(results.save_dir), "results.png")

img = mpimg.imread(results_png)
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Training Results", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Print final mAP@0.5 from training results
# results.results_dict contains the final epoch metrics
metrics = results.results_dict

print("=" * 40)
print("  Final Training Metrics")
print("=" * 40)
print(f"  mAP@0.5      : {metrics.get('metrics/mAP50(B)',    0):.4f}")
print(f"  mAP@0.5:0.95 : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"  Precision     : {metrics.get('metrics/precision(B)', 0):.4f}")
print(f"  Recall        : {metrics.get('metrics/recall(B)',    0):.4f}")
print("=" * 40)
print(f"  Best weights  : {os.path.abspath(best_dst)}")